# Nemotron Fine-tuning — Option 1+2+3 (v5 — 2600×2ep)

-  Label masking fixed: prompt tokens are masked (`-100`), loss is computed on answer tokens only
-  Data `2600` samples × 2 epochs (~1,300 optimizer steps, fits within 9 hours), epochs set to `2` (~2,000 optimizer steps)
-  — Cosine LR scheduler with 10% warmup
**Bug fixes v2:**
-  Critical fix: label masking now uses token IDs directly from the tokenized full sequence (not re-encoded strings) — prevents BPE tokenizer mismatch
- Fix: `model.device` replaced with `next(model.parameters()).device` for device_map="auto"
- Fix: `UnboundLocalError` guard on flush step if dataset is very small
- Fix: `total_steps` is recalculated after the dataset is fully built (not from SUBSAMPLE_SIZE)

Estimated runtime: ~4.5–5 hours on RTX Pro 6000.

In [ ]:
import sys, os

# ── Inject path dari nvidia-utility-script ke sys.path ──
# Utility script (kernelVersion 303833295) berisi mamba_ssm, torch patches, dll.
# Internet disabled di competition environment — pip install tidak bisa dipakai.
# mamba_ssm sudah tersedia via utility script ini.
UTIL_BASE = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script"

if os.path.isdir(UTIL_BASE):
    for subdir in ["", "torch", "mamba_ssm"]:
        p = os.path.join(UTIL_BASE, subdir)
        if os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)
    print(f"\u2705 Utility script di-inject ke sys.path: {UTIL_BASE}")
else:
    print(f"\u26a0\ufe0f  Utility script tidak ditemukan di {UTIL_BASE}")
    print("   Pastikan datasource kernelVersion 303833295 di-attach.")

try:
    import mamba_ssm
    print(f"\u2705 mamba_ssm ready")
except ImportError as e:
    print(f"\u26a0\ufe0f  mamba_ssm import failed: {e}")
    print("   Training mungkin tetap bisa jalan — model akan coba import sendiri.")


In [6]:
import os
import gc
import torch

# Prefer polars, fallback to pandas for local environments
try:
    import polars as pl
    DF_BACKEND = "polars"
except Exception:
    import pandas as pd
    DF_BACKEND = "pandas"

# -- Config --
SUBSAMPLE_SIZE = 2600
LOCAL_PREVIEW_ROWS = 64
TRAIN_CSV = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
IS_KAGGLE = os.path.isdir("/kaggle")


def sample_df(df, n, seed=42):
    if DF_BACKEND == "polars":
        return df.sample(n=min(n, len(df)), seed=seed)
    return df.sample(n=min(n, len(df)), random_state=seed)


def head_df(df, n=5):
    return df.head(n)


def iter_rows_named(df):
    if DF_BACKEND == "polars":
        return df.iter_rows(named=True)
    return df.to_dict(orient="records")


# -- Load train data (Kaggle if available, otherwise local demo fallback) --
if IS_KAGGLE and os.path.exists(TRAIN_CSV):
    if DF_BACKEND == "polars":
        train = pl.read_csv(TRAIN_CSV)
    else:
        train = pd.read_csv(TRAIN_CSV)

    print(f"Training samples total: {len(train)}")
    print(head_df(train))
    train = sample_df(train, SUBSAMPLE_SIZE, seed=42)
    print(f"Subsampled to: {len(train)} examples")
else:
    print("⚠️ Kaggle train.csv not found. Using local preview dataset.")
    base_rows = [
        {"prompt": "If x + 7 = 19, what is x?", "answer": "12"},
        {"prompt": "Compute 15 * 6.", "answer": "90"},
        {"prompt": "A triangle has angles 50 and 60. Find the third angle.", "answer": "70"},
        {"prompt": "Solve for y: 3y - 5 = 16", "answer": "7"},
        {"prompt": "What is 2^5?", "answer": "32"},
        {"prompt": "What is 1/4 as a decimal?", "answer": "0.25"},
        {"prompt": "Find gcd(18, 24).", "answer": "6"},
        {"prompt": "What is 12 percent of 50?", "answer": "6"},
    ]
    demo_rows = (base_rows * ((LOCAL_PREVIEW_ROWS + len(base_rows) - 1) // len(base_rows)))[:LOCAL_PREVIEW_ROWS]

    if DF_BACKEND == "polars":
        train = pl.DataFrame(demo_rows)
    else:
        train = pd.DataFrame(demo_rows)

    print(f"Local preview samples: {len(train)}")
    print(head_df(train))

# -- Model path (Kaggle) or None (local preview mode) --
MODEL_PATH = None
if IS_KAGGLE:
    try:
        import kagglehub

        MODEL_PATH = kagglehub.model_download(
            "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
        )
    except Exception as e:
        print(f"⚠️ kagglehub/model download unavailable: {e}")

OUTPUT_DIR = "/kaggle/working/adapter" if IS_KAGGLE else "./adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Data backend: {DF_BACKEND}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MODEL_PATH: {MODEL_PATH}")

⚠️ Kaggle train.csv not found. Using local preview dataset.
Local preview samples: 64
                                              prompt answer
0                          If x + 7 = 19, what is x?     12
1                                    Compute 15 * 6.     90
2  A triangle has angles 50 and 60. Find the thir...     70
3                           Solve for y: 3y - 5 = 16      7
4                                       What is 2^5?     32
Data backend: pandas
OUTPUT_DIR: ./adapter
MODEL_PATH: None


In [9]:
import shutil, os, stat, sys

LOCAL_PREVIEW_MODE = MODEL_PATH is None

if not LOCAL_PREVIEW_MODE:
    # Import training-only dependencies only when running full Kaggle training mode.
    from peft import LoraConfig, get_peft_model, TaskType
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from torch.utils.data import Dataset, DataLoader

    # -- Optional ptxas-blackwell setup (Kaggle utility-script path) --
    _ptxas_src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
    _ptxas_dst = "/tmp/ptxas-blackwell"

    try:
        shutil.copy2(_ptxas_src, _ptxas_dst)
        os.chmod(_ptxas_dst, os.stat(_ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        import triton.backends.nvidia as nv_backend

        src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
        dst_bin = "/tmp/triton_nvidia_bin"
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
        os.environ["TRITON_PTXAS_PATH"] = _ptxas_dst
        print("✅ ptxas-blackwell configured")

    except FileNotFoundError:
        print("⚠️ ptxas-blackwell tidak ditemukan — triton fallback ke ptxas default")
        print("   Training tetap dilanjutkan.")
    except Exception as e:
        print(f"⚠️ ptxas setup: {e} — melanjutkan dengan default")

LORA_RANK = 32
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 2
GRAD_ACCUM = 4
LR = 2e-4


def build_training_text(prompt: str, answer: str) -> str:
    user_msg = prompt + "\nPlease put your final answer inside \\boxed{}."
    assistant_msg = f"<think></think>\\boxed{{{answer}}}"

    # Use tokenizer chat template when available; otherwise use portable fallback format.
    if "tokenizer" in globals() and tokenizer is not None:
        try:
            messages = [
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": assistant_msg},
            ]
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            pass

    return (
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    )


if LOCAL_PREVIEW_MODE:
    print("⚠️ Local preview mode: skip model load + dataset tokenization.")
    model = None
    tokenizer = None
    TRAIN_DEVICE = torch.device("cpu")
else:
    # -- Load model/tokenizer for full training mode --
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

    # Force slow path — bypass broken CUDA kernels
    for name, mod in sys.modules.items():
        if "modeling_nemotron_h" in name:
            mod.is_fast_path_available = False
            print(f"Patched {name}: is_fast_path_available = False")

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"Model loaded. Vocab size: {len(tokenizer)}")

    TRAIN_DEVICE = next(model.parameters()).device
    print(f"Training device: {TRAIN_DEVICE}")

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=16,
        target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

rows_named = iter_rows_named(train)
texts = [
    build_training_text(row["prompt"], row["answer"])
    for row in rows_named
]
print(f"Built {len(texts)} training examples")
if texts:
    print(f"\nExample (first 600 chars):\n{texts[0][:600]}")

if not LOCAL_PREVIEW_MODE:
    class SFTDataset(Dataset):
        def __init__(self, texts, tokenizer, max_length):
            self.encodings = []
            skipped = 0

            im_start_ids = tokenizer.encode("<|im_start|>", add_special_tokens=False)
            assistant_ids = tokenizer.encode("assistant", add_special_tokens=False)

            print(f"<|im_start|> token IDs: {im_start_ids}")
            print(f"'assistant'  token IDs: {assistant_ids}")

            for text in texts:
                enc = tokenizer(
                    text,
                    truncation=True,
                    max_length=max_length,
                    padding="max_length",
                    return_tensors="pt",
                )
                ids = enc["input_ids"].squeeze(0)
                mask = enc["attention_mask"].squeeze(0)

                labels = ids.clone()
                labels[mask == 0] = -100

                ids_list = ids.tolist()
                search_seq = im_start_ids + assistant_ids
                search_len = len(search_seq)
                prompt_end = None

                for pos in range(len(ids_list) - search_len + 1):
                    if ids_list[pos: pos + search_len] == search_seq:
                        prompt_end = pos + search_len
                        break

                if prompt_end is None:
                    skipped += 1
                    continue

                labels[:prompt_end] = -100

                valid_tokens = (labels != -100).sum().item()
                if valid_tokens < 2:
                    skipped += 1
                    continue

                self.encodings.append(
                    {
                        "input_ids": ids,
                        "attention_mask": mask,
                        "labels": labels,
                    }
                )

            print(f"Tokenized {len(self.encodings)} examples (skipped {skipped})")
            if self.encodings:
                sample_labels = self.encodings[0]["labels"]
                n_masked = (sample_labels == -100).sum().item()
                n_active = (sample_labels != -100).sum().item()
                pct = n_active / (n_masked + n_active) * 100
                print(
                    f"Sample label distribution: {n_masked} masked (prompt+padding) | {n_active} active (answer) | {pct:.1f}% active"
                )

        def __len__(self):
            return len(self.encodings)

        def __getitem__(self, idx):
            return self.encodings[idx]

    dataset = SFTDataset(texts, tokenizer, MAX_SEQ_LEN)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
    assert len(dataset) > 0, "Dataset kosong! Periksa format teks dan tokenizer."
    print(f"Dataset ready: {len(dataset)} samples")
else:
    dataset = None
    dataloader = None
    print("Preview mode ready. You can run the training_text preview cell now.")

⚠️ Local preview mode: skip model load + dataset tokenization.
Built 64 training examples

Example (first 600 chars):
<|im_start|>user
If x + 7 = 19, what is x?
Please put your final answer inside \boxed{}.<|im_end|>
<|im_start|>assistant
<think></think>\boxed{12}<|im_end|>
Preview mode ready. You can run the training_text preview cell now.


In [10]:
import random

# Preview randomized training_text examples from the current train dataframe
N_EXAMPLES = 20
SEED = 42
rng = random.Random(SEED)

if "train" not in globals():
    raise RuntimeError("train dataframe not found. Run Cell 3 first.")

if "build_training_text" not in globals():
    raise RuntimeError("build_training_text not found. Run Cell 4 first.")

if hasattr(train, "to_dicts"):
    rows = train.to_dicts()
elif hasattr(train, "to_dict"):
    rows = train.to_dict(orient="records")
else:
    rows = list(train)

n_available = len(rows)
if n_available == 0:
    raise RuntimeError("train dataframe is empty.")

n_show = min(N_EXAMPLES, n_available)
indices = rng.sample(range(n_available), k=n_show)

print(f"Showing {n_show} randomized training_text examples (seed={SEED})")

for i, idx in enumerate(indices, start=1):
    row = rows[idx]
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    text = build_training_text(prompt, answer)

    print(f"\n{'=' * 22} Example {i} (row {idx}) {'=' * 22}")
    print(text)

Showing 20 randomized training_text examples (seed=42)

====================== Example 1 (row 14) ======================
<|im_start|>user
Find gcd(18, 24).
Please put your final answer inside \boxed{}.<|im_end|>
<|im_start|>assistant
<think></think>\boxed{6}<|im_end|>

====================== Example 2 (row 1) ======================
<|im_start|>user
Compute 15 * 6.
Please put your final answer inside \boxed{}.<|im_end|>
<|im_start|>assistant
<think></think>\boxed{90}<|im_end|>

====================== Example 3 (row 47) ======================
<|im_start|>user
What is 12 percent of 50?
Please put your final answer inside \boxed{}.<|im_end|>
<|im_start|>assistant
<think></think>\boxed{6}<|im_end|>

====================== Example 4 (row 17) ======================
<|im_start|>user
Compute 15 * 6.
Please put your final answer inside \boxed{}.<|im_end|>
<|im_start|>assistant
<think></think>\boxed{90}<|im_end|>

====================== Example 5 (row 15) ======================
<|im_start|>user
W

In [12]:
import torch, torch.nn.functional as F, sys
from transformers import get_cosine_schedule_with_warmup

# ── Bypass Triton rmsnorm ──
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# ── Training setup ──
model.train()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.01,
)

# ── [BUG FIX v2] total_steps dihitung dari len(dataset) AKTUAL ──
# (bukan SUBSAMPLE_SIZE, karena beberapa sample mungkin di-skip saat tokenisasi)
actual_n    = len(dataset)
total_steps = (actual_n * NUM_EPOCHS) // GRAD_ACCUM

# ── [OPSI 3] Cosine LR scheduler dengan 10% warmup ──
warmup_steps = max(1, total_steps // 10)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Dataset size (actual): {actual_n}")
print(f"Training: {NUM_EPOCHS} epochs, ~{total_steps} optimizer steps")
print(f"Scheduler: cosine warmup ({warmup_steps} warmup steps / {total_steps} total)")
print(f"LR: 0 → {LR:.1e} (warmup) → ~0 (cosine decay)")

for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    optimizer.zero_grad()
    last_i = -1  # [BUG FIX v2] guard untuk UnboundLocalError

    for i, batch in enumerate(dataloader):
        # [BUG FIX v2] gunakan TRAIN_DEVICE yang sudah di-resolve sebelumnya
        batch  = {k: v.to(TRAIN_DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss    = outputs.loss / GRAD_ACCUM
        loss.backward()
        running_loss += outputs.loss.item()
        last_i = i

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            step = (i + 1) // GRAD_ACCUM
            if step % 50 == 0:
                avg        = running_loss / (i + 1)
                current_lr = scheduler.get_last_lr()[0]
                print(f"  epoch {epoch+1} | step {step}/{total_steps} | avg_loss {avg:.4f} | lr {current_lr:.2e}")

    # ── Flush sisa gradient (jika batch terakhir tidak genap GRAD_ACCUM) ──
    # [BUG FIX v2] guard dengan last_i — tidak akan crash jika dataset kosong
    if last_i >= 0 and (last_i + 1) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    avg_loss   = running_loss / max(1, last_i + 1)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} complete — avg loss: {avg_loss:.4f} | lr: {current_lr:.2e}")


c:\Users\asvir\Envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'NoneType' object has no attribute 'train'

In [ ]:
import zipfile

model.save_pretrained(OUTPUT_DIR)
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("✓ submission.zip looks correct")
